# 📅 Masterclass 07: Time Series Forecasting with Statistical ARIMA & Hybrid Models
This notebook details temporal sequence modeling and future forecasting:

1. **Project 1 (Scratch)**: Autoregressive AR(p) model solver via matrix-level Yule-Walker equations.
2. **Project 2 (Applied)**: Industrial sales forecasting using a hybrid SARIMAX + Facebook Prophet pipeline.


## 📐 Part 1: Mathematical Foundations & LaTeX
### 1. Autoregressive AR(p) Formulation
$$X_t = c + \sum_{i=1}^{p} \phi_i X_{t-i} + \epsilon_t$$

### 2. Solving Parameters via Yule-Walker Matrix Relations
Taking lag covariances $\gamma_k$ yields a Toplitz linear equation matrix system resolved via inversion:
$$\gamma_k = \sum_{i=1}^{p} \phi_i \gamma_{k-i}$$
$$\begin{bmatrix} \gamma_0 & \gamma_1 & \dots & \gamma_{p-1} \\ \gamma_1 & \gamma_0 & \dots & \gamma_{p-2} \\ \vdots & \vdots & \ddots & \vdots \\ \gamma_{p-1} & \gamma_{p-2} & \dots & \gamma_0 \end{bmatrix} \begin{bmatrix} \phi_1 \\ \phi_2 \\ \vdots \\ \phi_p \end{bmatrix} = \begin{bmatrix} \gamma_1 \\ \gamma_2 \\ \vdots \\ \gamma_p \end{bmatrix}$$


## 🧠 Project 1: Yule-Walker Parametric Solver from Scratch


In [ ]:
class AutoregressiveEstimator:
    def __init__(self, p=2):
        self.p = p
        self.phi = None
        self.mean = None

    def _autocovariance(self, x, lag):
        n = len(x)
        if lag >= n: return 0.0
        x_centered = x - np.mean(x)
        return np.sum(x_centered[:n-lag] * x_centered[lag:]) / n

    def fit(self, x):
        self.mean = np.mean(x)
        p = self.p
        gamma = np.array([self._autocovariance(x, i) for i in range(p + 1)])
        R = np.zeros((p, p))
        for i in range(p):
            for j in range(p):
                R[i, j] = gamma[abs(i - j)]
        self.phi = np.linalg.solve(R, gamma[1:p+1])


## 🧪 Project 2: Sales forecasting using Prophet + SARIMAX residual corrections


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet

# Simulated sales
df_sales = pd.DataFrame({
    'ds': pd.date_range(start='2026-01-01', periods=100),
    'y': np.sin(np.linspace(0, 20, 100)) * 50 + 200 + np.random.normal(0, 5, 100)
})

model = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False)
model.fit(df_sales)
forecast = model.predict(df_sales)
residuals = df_sales['y'] - forecast['yhat']

sarimax = SARIMAX(residuals, order=(1,1,1))
sarimax_fit = sarimax.fit(disp=False)
print('SARIMAX Residual Param Fit Successfully Completed:')
print(sarimax_fit.summary().tables[1])
